# Amazon Product Recommendation System
### Group 8 — TBS Education

| Member |
|---|
| EL MIR Otmane |
| SAVOYE Raphael |
| MOUMNI Youssef |

---
**Objective:** Build three recommendation engines on 20,000 Amazon Beauty & Personal Care reviews:
1. **Popularity-Based** — best products globally (handles cold-start)
2. **Item-Based CF** — products similar to what the user already liked
3. **User-Based CF** — products liked by similar users

**Dataset:** `Group8-2.xlsx` — place in the same folder as this notebook.


## 1. Imports & Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

%matplotlib inline
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
print('All libraries loaded successfully.')

## 2. Data Loading

We load the Amazon reviews dataset from an Excel file using `pd.read_excel()`.
The file must be placed in the **same folder** as this notebook.


In [ ]:
df = pd.read_excel('Group8-2.xlsx')
print(f'Shape     : {df.shape}')
print(f'Columns   : {list(df.columns)}')
df.head()

## 3. Exploratory Data Analysis (EDA)

Before building any model, we must understand the data — this step is called EDA (Exploratory Data Analysis).


### 3.1 Key statistics


In [ ]:
print(f"Total reviews  : {len(df):,}")
print(f"Unique users   : {df['UserId'].nunique():,}")
print(f"Unique products: {df['ProductId'].nunique():,}")
print(f"Average rating : {df['Rating'].mean():.2f} / 5")
print(f"Rating range   : {df['Rating'].min()} — {df['Rating'].max()}")
print(f"\nMissing values:\n{df.isnull().sum()}")

### 3.2 Rating distribution — J-Curve Bias


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

rating_counts = df['Rating'].value_counts().sort_index()
colors = ['#E74C3C','#E67E22','#F1C40F','#2ECC71','#1ABC9C']

# Bar chart
axes[0].bar(rating_counts.index, rating_counts.values, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Rating Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Stars'); axes[0].set_ylabel('Number of reviews')
for i, v in enumerate(rating_counts.values):
    axes[0].text(rating_counts.index[i], v + 80, f'{v:,}', ha='center', fontsize=9)

# Pie chart
axes[1].pie(rating_counts.values, labels=[f'{i}★' for i in rating_counts.index],
            colors=colors, autopct='%1.1f%%', startangle=140, textprops={'fontsize':10})
axes[1].set_title('Rating Share (%)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_rating_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n>>> {rating_counts[5]/len(df)*100:.1f}% of all reviews are 5-star — J-Curve Bias')

**J-Curve Bias:** 60% of reviews are 5 stars. People mostly rate when they love a product. A simple average is therefore unreliable — this motivates our **Bayesian score** in Section 5.


### 3.3 Sparsity of the rating matrix


In [ ]:
n_users    = df['UserId'].nunique()
n_products = df['ProductId'].nunique()
sparsity   = 1 - len(df) / (n_users * n_products)

print(f'Users            : {n_users:,}')
print(f'Products         : {n_products:,}')
print(f'Actual ratings   : {len(df):,}')
print(f'Max possible     : {n_users * n_products:,}')
print(f'Sparsity         : {sparsity:.4%}')
print(f'\n>>> {sparsity:.2%} of all user-product cells are EMPTY')
print('>>> This is the core challenge of collaborative filtering')

**Sparsity = 99.97%** — Almost no user has rated more than a tiny fraction of all products. This is why we filter to users/products with ≥3 ratings in Section 4.


### 3.4 Top products by review volume


In [ ]:
product_stats = (
    df.groupby(['ProductId','product_name'])['Rating']
      .agg(avg_rating='mean', num_ratings='count')
      .reset_index()
      .sort_values('num_ratings', ascending=False)
)

top10 = product_stats.head(10)
fig, ax = plt.subplots(figsize=(12, 5))
colors_bar = plt.cm.Blues(np.linspace(0.4, 0.9, 10))[::-1]
bars = ax.barh(top10['product_name'].str[:45], top10['num_ratings'], color=colors_bar)
ax.set_xlabel('Number of ratings')
ax.set_title('Top 10 Most Reviewed Products', fontsize=13, fontweight='bold')
ax.invert_yaxis()
for bar, val in zip(bars, top10['num_ratings']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('fig_top10.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Preparation

### 4.1 Filter active users and products

We keep only users who rated **≥ 3 products** and products with **≥ 3 ratings**.
A user with 1 rating gives no information about their preferences.
The threshold 3 is a trade-off: enough signal without losing too much data.


In [ ]:
user_counts  = df.groupby('UserId')['Rating'].count()
prod_counts  = df.groupby('ProductId')['Rating'].count()
active_users = user_counts[user_counts >= 3].index
active_prods = prod_counts[prod_counts >= 3].index

df_cf = df[df['UserId'].isin(active_users) & df['ProductId'].isin(active_prods)].copy()

print(f'Before : {len(df):,} reviews | {df["UserId"].nunique():,} users | {df["ProductId"].nunique():,} products')
print(f'After  : {len(df_cf):,} reviews | {df_cf["UserId"].nunique():,} users | {df_cf["ProductId"].nunique():,} products')
print(f'Kept   : {len(df_cf)/len(df)*100:.1f}% of original data')

### 4.2 Build the user-item rating matrix


In [ ]:
# Rows = users | Columns = products | Values = star rating (0 if not rated)
rating_matrix = (
    df_cf.pivot_table(index='UserId', columns='ProductId', values='Rating')
         .fillna(0)
)
# NOTE: 0 means 'not yet rated' — NOT 'zero stars'

print(f'Matrix shape     : {rating_matrix.shape}  ({rating_matrix.shape[0]} users x {rating_matrix.shape[1]} products)')
new_sp = 1 - (rating_matrix > 0).sum().sum() / (rating_matrix.shape[0]*rating_matrix.shape[1])
print(f'Sparsity (after) : {new_sp:.4%}')
rating_matrix.head(3)

## 5. System 1 — Popularity-Based Recommender

### How it works
Recommends the **same top-N products to every user**, ranked by a **Bayesian weighted score**.
This corrects for the J-Curve bias: a product with 1 review at 5★ should not rank above
one with 500 reviews at 4.8★.

**Bayesian formula:**

$$score = \\frac{n}{n+m} \\times \\overline{r} + \\frac{m}{n+m} \\times C$$

- $n$ = number of ratings for this product
- $m$ = minimum threshold (we use 5)
- $\\overline{r}$ = product's average rating
- $C$ = global mean rating across all products (~4.19)

**Advantage:** solves the **cold-start problem** — works even for brand new users with zero history.


In [ ]:
C = df_cf['Rating'].mean()          # Global mean
m = 5                                # Minimum ratings threshold

pop_stats = (
    df_cf.groupby(['ProductId','product_name'])['Rating']
         .agg(avg_rating='mean', num_ratings='count')
         .reset_index()
)

# Bayesian weighted score
pop_stats['score'] = (
    (pop_stats['num_ratings'] / (pop_stats['num_ratings'] + m)) * pop_stats['avg_rating']
  + (m               / (pop_stats['num_ratings'] + m)) * C
)

pop_stats = pop_stats.sort_values('score', ascending=False)

print(f'Global mean C = {C:.4f}')
print(f'Threshold   m = {m}')
print('\nTop 5 products by Bayesian score:')
pop_stats.head(5)[['product_name','avg_rating','num_ratings','score']]

In [ ]:
def popularity_recommender(n=5):
    """Return top-n products by Bayesian score. Same for every user."""
    return pop_stats.head(n)[['product_name','ProductId','avg_rating','num_ratings','score']].reset_index(drop=True)

# Test
print('--- Popularity-Based Top 5 (same for every user) ---')
popularity_recommender(5)

### 5.1 Visualise top 5 popularity picks


In [ ]:
pop_top5 = popularity_recommender(5)

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.barh(pop_top5['product_name'].str[:40], pop_top5['score'],
               color=plt.cm.Blues(np.linspace(0.5, 0.9, 5))[::-1])
ax.set_xlabel('Bayesian Score')
ax.set_title('Top 5 Products — Popularity-Based (Bayesian Score)', fontsize=12, fontweight='bold')
ax.set_xlim(0, pop_top5['score'].max() * 1.12)
ax.invert_yaxis()
for bar, row in zip(bars, pop_top5.itertuples()):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{row.score:.4f}  ({row.num_ratings} ratings)', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('fig_popularity.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. System 2 — Item-Based Collaborative Filtering

### How it works
For a given user, find products with **similar rating patterns** to those already rated,
and recommend the 5 most similar ones the user hasn't seen yet.

Similarity is measured using **cosine similarity** between product rating vectors:

$$sim(X, Y) = \\frac{X \\cdot Y}{\\|X\\| \\cdot \\|Y\\|}$$

Each product is a **vector** of all ratings it received from users.
Two products rated similarly by the same users point in the same direction → high similarity.


In [ ]:
# Transpose: rows = products, columns = users
# cosine_similarity computes similarity between rows
item_sim    = cosine_similarity(rating_matrix.T)
item_sim_df = pd.DataFrame(item_sim,
                           index=rating_matrix.columns,
                           columns=rating_matrix.columns)
print(f'Item similarity matrix: {item_sim_df.shape}  (product x product)')
print(f'Example similarity    : {item_sim_df.iloc[0,1]:.4f}')

In [ ]:
def item_based_recommender(user_id, rating_matrix, item_sim_df, df_lookup, n=5):
    """Recommend n products most similar to those already rated by user_id."""
    if user_id not in rating_matrix.index:
        return f'User {user_id} not found.'

    user_ratings   = rating_matrix.loc[user_id]
    rated_products = user_ratings[user_ratings > 0].index.tolist()

    if not rated_products:
        return 'This user has no ratings.'

    # For each unrated product: average cosine similarity to all rated products
    unrated = user_ratings[user_ratings == 0].index
    scores  = {p: item_sim_df.loc[p, rated_products].mean() for p in unrated}

    top_recs   = pd.Series(scores).sort_values(ascending=False).head(n)
    id_to_name = df_lookup.drop_duplicates('ProductId').set_index('ProductId')['product_name']

    result = top_recs.reset_index()
    result.columns = ['ProductId','similarity_score']
    result['product_name']     = result['ProductId'].map(id_to_name)
    result['similarity_score'] = result['similarity_score'].round(4)
    result.index = result.index + 1
    return result[['product_name','ProductId','similarity_score']]


# Test
sample_user = rating_matrix.index[0]
print(f'Sample user: {sample_user}')
print('\n--- Products already rated ---')
already = df_cf[df_cf['UserId']==sample_user][['product_name','Rating']].sort_values('Rating',ascending=False)
print(already.to_string(index=False))
print('\n--- Item-Based Recommendations ---')
item_based_recommender(sample_user, rating_matrix, item_sim_df, df_cf)

### 6.1 Heatmap — Item Similarity (Top 15 products)


In [ ]:
top15_prods = df_cf['ProductId'].value_counts().head(15).index
sub_item    = item_sim_df.loc[top15_prods, top15_prods]
name_map    = df_cf.drop_duplicates('ProductId').set_index('ProductId')['product_name']
labels      = [name_map.get(p, p)[:22] for p in top15_prods]

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(sub_item, xticklabels=labels, yticklabels=labels,
            cmap='YlOrRd', linewidths=0.3, ax=ax, vmin=0, vmax=1)
ax.set_title('Item-Item Cosine Similarity Heatmap (Top 15 Products)', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('fig_item_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. System 3 — User-Based Collaborative Filtering

### How it works
Find the **k=10 users** with the most similar rating patterns to the target user,
then recommend products most enjoyed by those neighbours that the target hasn't seen.

**Critical detail:** we call `.drop(user_id)` before sorting.
Why? The most similar user to anyone is always themselves (similarity = 1.0).
Without dropping self, we'd use the target as their own neighbour — circular and useless.


In [ ]:
user_sim    = cosine_similarity(rating_matrix)
user_sim_df = pd.DataFrame(user_sim,
                           index=rating_matrix.index,
                           columns=rating_matrix.index)
print(f'User similarity matrix: {user_sim_df.shape}  (user x user)')

In [ ]:
def user_based_recommender(user_id, rating_matrix, user_sim_df, df_lookup, n_neighbors=10, n=5):
    """Recommend n products liked by the n_neighbors most similar users."""
    if user_id not in rating_matrix.index:
        return f'User {user_id} not found.'

    user_ratings  = rating_matrix.loc[user_id]
    already_rated = user_ratings[user_ratings > 0].index

    # .drop(user_id) — exclude self (always similarity = 1.0)
    similar_users = (
        user_sim_df[user_id].drop(user_id)
                            .sort_values(ascending=False)
                            .head(n_neighbors).index
    )

    # Average rating from neighbours for products NOT yet seen by target user
    mean_neighbor   = rating_matrix.loc[similar_users].mean(axis=0)
    recommendations = mean_neighbor.drop(already_rated).sort_values(ascending=False).head(n)

    id_to_name = df_lookup.drop_duplicates('ProductId').set_index('ProductId')['product_name']
    result = recommendations.reset_index()
    result.columns = ['ProductId','predicted_rating']
    result['product_name']     = result['ProductId'].map(id_to_name)
    result['predicted_rating'] = result['predicted_rating'].round(4)
    result.index = result.index + 1
    return result[['product_name','ProductId','predicted_rating']]


print(f'User: {sample_user}')
print('\n--- User-Based Recommendations ---')
user_based_recommender(sample_user, rating_matrix, user_sim_df, df_cf)

### 7.1 Heatmap — User Similarity (Top 20 active users)


In [ ]:
top20_users = df_cf['UserId'].value_counts().head(20).index
sub_user    = user_sim_df.loc[top20_users, top20_users]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(sub_user, cmap='Blues', linewidths=0.2, ax=ax, vmin=0, vmax=1,
            xticklabels=False, yticklabels=False)
ax.set_title('User-User Cosine Similarity Heatmap (Top 20 Active Users)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_user_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Comparison & Business Recommendations

### 8.1 Side-by-side for one user


In [ ]:
print('='*65)
print('POPULARITY-BASED  (same for every user)')
print('='*65)
print(popularity_recommender(5)[['product_name','score']].to_string())

print('\n'+'='*65)
print(f'ITEM-BASED  (personalised for {sample_user[:20]}...)')
print('='*65)
print(item_based_recommender(sample_user, rating_matrix, item_sim_df, df_cf)
      [['product_name','similarity_score']].to_string())

print('\n'+'='*65)
print(f'USER-BASED  (personalised for {sample_user[:20]}...)')
print('='*65)
print(user_based_recommender(sample_user, rating_matrix, user_sim_df, df_cf)
      [['product_name','predicted_rating']].to_string())

### 8.2 Comparison table

| Criterion | Popularity-Based | Item-Based CF | User-Based CF |
|---|---|---|---|
| **Personalised?** | No — same for all | Yes | Yes |
| **Cold-start** | Handled (no history needed) | Needs history | Needs history |
| **Sparsity robustness** | Not affected | Moderate | Low |
| **Scalability** | O(1) — instant | Pre-computable | O(n²) per user |
| **Serendipity** | Low | Medium | High |
| **Production ready** | Yes | Yes (Amazon) | Not at scale |

### 8.3 Business Recommendation

**Hybrid strategy:**
1. **New users (0-2 ratings)** → Popularity-Based (cold-start)
2. **Users with 3+ ratings** → Item-Based CF (personalised, scalable)
3. **Highly active users** → User-Based CF (serendipitous discovery)


---
## Notable Woman Scientist: Fei-Fei Li (born 1976)

**Fei-Fei Li** is a Chinese-American computer scientist, Professor at Stanford University,
and co-director of the Stanford Human-Centered AI Institute (HAI).
She created **ImageNet** (2009) — 14 million labelled images that launched modern deep learning.
Relevant to our project: the same principle of learning from large-scale user data
that she pioneered in vision applies directly to recommendation systems.


---
## References

- Sarwar, B. et al. (2001). *Item-based collaborative filtering recommendation algorithms.* WWW '01.
- Resnick, P. et al. (1994). *GroupLens: An open architecture for collaborative filtering.* CSCW '94.
- Amazon dataset: McAuley, J. & Leskovec, J. (2013). *Hidden factors and hidden topics.* RecSys '13.
- Fei-Fei Li et al. (2009). *ImageNet: A large-scale hierarchical image database.* CVPR '09.
